# 09 · EDA y estadística con pandas

**Tema 13 · Notebook guiado de Big Data.** Usa este cuaderno en paralelo con el [08 en PySpark](08_data_preparation_and_analysis.ipynb). Ambos siguen la misma encuesta, las mismas reglas de limpieza y los mismos contrastes. Ejecuta las celdas de arriba abajo. Cada paso explica qué queremos averiguar, cómo lo hacemos y cómo leer la salida. Trabajaremos con tablas y resultados numéricos.

**Pregunta de la demostración.** ¿Cuánto difieren los salarios mensuales medios de los grupos profesionales A y B? Después veremos una pregunta sobre categorías y otra sobre la relación entre dos medidas numéricas.

Los archivos `empleados_curso.csv` y `ciudades_contexto.csv` están en `PySpark/data`. Puedes iniciar el kernel desde la raíz del repositorio, `PySpark` o `PySpark/notebooks`. Necesitas pandas y SciPy 1.11 o posterior. Si faltan, ejecuta `%pip install pandas "scipy>=1.11"` en una celda y reinicia el kernel si Jupyter lo solicita.

La encuesta es una versión reducida del sprint 8. Los nombres y DNI se retiraron. `id_persona` es un código local para reconocer filas de una misma persona. `id_registro` identifica copias exactas en el archivo original. Las tablas de clase usan resultados agregados.

## 1 · Entender qué tenemos

El análisis exploratorio de datos (EDA) nos ayuda a entender qué representa cada fila, detectar problemas y decidir si los datos permiten responder la pregunta. Aquí esperamos una fila por persona una vez revisadas las repeticiones.

Empezamos por las dimensiones, los tipos y los valores faltantes. Los porcentajes ayudan a comparar columnas con la misma base.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

inicio = Path.cwd()
archivos = ('empleados_curso.csv', 'ciudades_contexto.csv')
candidatas = (inicio / 'data', inicio.parent / 'data', inicio / 'PySpark' / 'data')
carpeta = next((ruta for ruta in candidatas
                if all((ruta / archivo).is_file() for archivo in archivos)), None)
if carpeta is None:
    raise FileNotFoundError('Abre el notebook desde el repositorio, PySpark o PySpark/notebooks.')
original = pd.read_csv(carpeta / 'empleados_curso.csv', dtype={'id_persona': 'Int64'})
print(f'{len(original)} filas y {original.shape[1]} columnas')
display(original.drop(columns=['id_persona', 'id_registro']).head(5))

perfil = pd.DataFrame({
    'tipo': original.dtypes.astype(str),
    'nulos': original.isna().sum(),
    '% nulos': original.isna().mean().mul(100).round(1),
})
display(perfil)

**Lectura.** Un salario puede aparecer como texto por el símbolo de moneda o por los separadores. Un nulo en kilometraje también puede tener una explicación válida si la persona no tiene coche. Investigamos el motivo antes de rellenar valores.

## 2 · Evitar contar dos veces a una persona

Primero quitamos las copias exactas. Después distinguimos las filas sin identificador de las personas que conservan varias fichas diferentes. Para esta demostración apartamos todas las fichas conflictivas porque no tenemos una fecha fiable que indique cuál es la vigente. Guardamos el número de filas descartadas en cada paso.

In [ ]:
sin_copias = original.drop_duplicates(subset='id_registro')
identificados = sin_copias.dropna(subset=['id_persona'])
en_conflicto = identificados.duplicated(subset='id_persona', keep=False)
base = identificados.loc[~en_conflicto].copy()

auditoria = pd.Series({
    'filas de entrada': len(original),
    'copias exactas retiradas': len(original) - len(sin_copias),
    'filas sin código de persona': len(sin_copias) - len(identificados),
    'filas con fichas conflictivas': int(en_conflicto.sum()),
    'filas para explorar': len(base),
}, name='filas')
display(auditoria)
assert base['id_persona'].is_unique

## 3 · Pasar de texto a valores comparables

Corregimos grafías de ciudades presentes en el archivo y usamos los grupos definidos de A a D. En el salario retiramos marcas de moneda, interpretamos separadores según el patrón del archivo y traducimos tres importes escritos en catalán. Los valores ambiguos, vacíos o no positivos quedan sin importe utilizable.

En kilometraje, los negativos son inválidos. Si la persona declara que no tiene coche, el kilometraje queda como **no aplicable**, incluso si el archivo traía un número. Conservamos el original para auditar.

In [ ]:
grafias = {
    'ValÃ¨ncia': 'València', 'MÃ\xa0laga': 'Málaga', 'A CoruÃ±a': 'A Coruña',
    'DÃ¼sseldorf': 'Düsseldorf', 'MÃ¼nchen': 'München',
    'TromsÃ¸': 'Tromsø', 'KÃ¶ln': 'Köln',
}
base['ciudad'] = (base['ciudad'].astype('string').str.strip()
                  .replace(grafias).replace('Desconeguda', pd.NA))
grupos_validos = ['Grup A', 'Grup B', 'Grup C', 'Grup D']
base['grupo'] = base['grupo'].astype('string').str.strip()
base['grupo'] = base['grupo'].where(base['grupo'].isin(grupos_validos))

importe = (base['salario_original'].astype('string').str.lower()
           .str.replace(r'â‚¬|€|euros?', '', regex=True).str.strip()
           .replace({'mil vuit-cents': '1800', 'dos mil': '2000', 'tres mil': '3000'}))
# En este archivo, un separador seguido de tres cifras marca miles.
importe = importe.str.replace(r'(?<=\d)[.,](?=\d{3}(?:[.,]|$))', '', regex=True)
importe = importe.str.replace(',', '.', regex=False)
importe = importe.where(importe.str.fullmatch(r'[+-]?\d+(?:\.\d{1,2})?', na=False))
salario = pd.to_numeric(importe, errors='coerce')
base['salario_eur'] = salario.where(salario.gt(0))

coche = base['tiene_coche'].astype('string').str.lower().map({'true': True, 'false': False})
base['tiene_coche'] = coche.astype('boolean')
sin_coche = base['tiene_coche'].eq(False).fillna(False)
km_invalido = base['km_anuales'].lt(0) | sin_coche
base['km_limpios'] = base['km_anuales'].mask(km_invalido)

display(pd.Series({
    'grupo sin categoría válida': base['grupo'].isna().sum(),
    'salario sin importe utilizable': base['salario_eur'].isna().sum(),
    'km anotados pese a declarar que no tiene coche': (sin_coche & base['km_anuales'].notna()).sum(),
}, name='filas'))

## 4 · Describir antes de contrastar

Miramos cuántas personas hay por grupo, cuántas tienen salario utilizable y cómo se distribuyen esos importes. La media usa todos los valores. La mediana es el valor central. Los percentiles 25 y 75 delimitan la mitad central. La desviación típica resume cuánto se separan los importes de su media.

Compara la cobertura entre grupos. Si faltan más salarios en uno de ellos, el análisis de casos observados puede quedar sesgado.

In [ ]:
resumen = (base.groupby('grupo')['salario_eur']
           .agg(personas='size', observados='count', media='mean', mediana='median',
                desviacion='std', minimo='min', maximo='max'))
cuartiles = base.groupby('grupo')['salario_eur'].quantile([0.25, 0.75]).unstack()
resumen[['p25', 'p75']] = cuartiles
resumen['% sin salario'] = (1 - resumen['observados'] / resumen['personas']) * 100
display(resumen.round(2))

## 5 · Nivel 2 breve. Añadir contexto por ciudad

La misma encuesta se puede enriquecer con un índice de coste de vida. El archivo de contexto contiene ciudades homónimas. Como la encuesta no incluye un país de residencia fiable para esta unión, apartamos las claves ambiguas. El país de origen de una persona no identifica dónde vive.

Usamos una unión izquierda para conservar todas las personas. Si falta un índice, dejamos ese valor vacío. Este contexto sirve para generar preguntas y queda fuera de los dos contrastes.

In [ ]:
import unicodedata

def clave_ciudad(valor):
    if pd.isna(valor):
        return pd.NA
    # Corrige el literal de escape presente en una ciudad del archivo de contexto.
    valor = valor.replace(r'\u00f1', 'ñ')
    letras = unicodedata.normalize('NFKD', valor.strip().casefold())
    return ''.join(letra for letra in letras if not unicodedata.combining(letra))

contexto = pd.read_csv(carpeta / 'ciudades_contexto.csv')
contexto['clave_ciudad'] = contexto['ciudad'].map(clave_ciudad)
contexto = contexto.dropna(subset='clave_ciudad').drop_duplicates()
ambiguas = contexto.duplicated(subset='clave_ciudad', keep=False)
display(contexto.loc[ambiguas, ['ciudad', 'pais_contexto']])

contexto_unico = contexto.loc[~ambiguas, ['clave_ciudad', 'indice_coste_vida']]
ampliada = (base.assign(clave_ciudad=base['ciudad'].map(clave_ciudad))
            .merge(contexto_unico, on='clave_ciudad', how='left', validate='many_to_one'))
assert len(ampliada) == len(base)
print(f"Personas con contexto: {ampliada['indice_coste_vida'].notna().sum()} de {len(ampliada)}")

## 6 · Demostración 1. Comparar dos medias con Welch

**Pregunta.** ¿Difiere el salario medio mensual de los grupos A y B? Elegimos esos dos grupos para la demostración y calculamos A menos B.

**Hipótesis nula (H₀).** Las medias poblacionales son iguales. La alternativa contempla diferencias en ambos sentidos. Usamos un umbral del 5 % para ilustrar la decisión.

Welch admite varianzas diferentes. El contraste supone observaciones independientes. Verificamos una fila por persona y revisamos tamaños, extremos y dispersión con la tabla anterior. Una fila por persona evita el doble recuento, pero no demuestra independencia si hay, por ejemplo, personas agrupadas por equipos. Usamos salarios observados, sin sustituir ausencias por una media. La inferencia depende también del diseño de la encuesta.

In [ ]:
from scipy import stats

alpha = 0.05
a = base.loc[base['grupo'].eq('Grup A'), 'salario_eur'].dropna().astype(float)
b = base.loc[base['grupo'].eq('Grup B'), 'salario_eur'].dropna().astype(float)

welch = stats.ttest_ind(a, b, equal_var=False, alternative='two-sided')
intervalo = welch.confidence_interval(confidence_level=0.95)
diferencia = a.mean() - b.mean()

print(f'Salarios observados: A = {len(a)}, B = {len(b)}')
print(f'Media A: {a.mean():.2f} € · Media B: {b.mean():.2f} €')
print(f'Diferencia A − B: {diferencia:.2f} €')
print(f'IC del 95 %: [{intervalo.low:.2f}, {intervalo.high:.2f}] €')
print(f'Valor p: {welch.pvalue:.4g}')
print('Decisión al 5 %:', 'rechazamos H₀' if welch.pvalue < alpha else 'no rechazamos H₀')

**Cómo leerlo.** Empieza por el signo y el tamaño de la diferencia en euros. Después mira el intervalo para valorar la precisión. El valor p resume lo inusual que sería observar una diferencia estandarizada al menos tan extrema bajo H₀ y los supuestos del test. Si no rechazamos H₀, la evidencia resulta insuficiente para distinguir las medias con este criterio.

La comparación es exploratoria. El cargo, la ubicación o la selección de quienes respondieron podrían influir en las cifras. La encuesta permite describir asociaciones y no atribuir causas.

## 7 · Demostración 2. Comparar categorías con chi-cuadrado

**Pregunta.** ¿Se relaciona el grupo profesional con declarar que se tiene coche? Usamos los grupos A y B y mostramos las proporciones sobre respuestas válidas.

**H₀.** Las dos variables son independientes. El test compara los recuentos observados con los esperados bajo H₀. Revisamos que las celdas tengan al menos cinco observaciones y cinco casos esperados como regla práctica para esta aproximación.

In [ ]:
grupos_ab = base.loc[base['grupo'].isin(['Grup A', 'Grup B'])]
print('Respuestas sobre coche que faltan por grupo')
display(grupos_ab.groupby('grupo')['tiene_coche'].apply(lambda valores: valores.isna().sum()))

respuestas = grupos_ab.dropna(subset=['tiene_coche'])
tabla = pd.crosstab(respuestas['grupo'], respuestas['tiene_coche'])
tabla = tabla.rename(columns={False: 'No', True: 'Sí'})
display(tabla)
display(tabla.div(tabla.sum(axis=1), axis=0).mul(100).round(1))

if tabla.shape != (2, 2):
    print('La tabla necesita dos grupos y dos respuestas para este contraste.')
else:
    chi = stats.chi2_contingency(tabla, correction=False)
    esperados = pd.DataFrame(chi.expected_freq, index=tabla.index, columns=tabla.columns)
    display(esperados.round(1))
    if tabla.to_numpy().min() >= 5 and esperados.to_numpy().min() >= 5:
        brecha = (tabla['Sí'] / tabla.sum(axis=1)).loc['Grup A'] - (tabla['Sí'] / tabla.sum(axis=1)).loc['Grup B']
        print(f'Diferencia A − B: {100 * brecha:.1f} puntos porcentuales')
        print(f'Chi-cuadrado: {chi.statistic:.3f} · Valor p: {chi.pvalue:.4g}')
        print('Decisión al 5 %:', 'rechazamos H₀' if chi.pvalue < alpha else 'no rechazamos H₀')
    else:
        print('La aproximación puede fallar con recuentos pequeños.')

**Cómo leerlo.** Mira qué proporción declaró tener coche en cada grupo y cuántas respuestas faltan. El test evalúa la asociación bajo sus supuestos. Las proporciones ayudan a valorar su tamaño. Los contrastes del cuaderno tienen un fin didáctico. Si quisiéramos confirmar varias hipótesis, planificaríamos cómo controlar los falsos positivos.

## 8 · Demostración 3. ¿Cambian juntas dos medidas?

**Pregunta.** Entre las personas que declaran tener coche, ¿quienes ganan más tienden a recorrer más kilómetros al año?

La **correlación** resume la dirección y la intensidad de una relación. El coeficiente va de −1 a +1. Si es positivo, valores altos de una variable tienden a acompañar valores altos de la otra. Si es negativo, tienden a acompañar valores bajos.

- **Pearson** mide en qué grado la relación sigue una línea recta. Los valores extremos pueden influir en el resultado.
- **Spearman** ordena cada variable de menor a mayor y compara esas posiciones. Puede captar una tendencia que sube o baja de forma consistente aunque no siga una recta.

Calculamos ambos sobre las mismas personas. El salario es el importe observado y el kilometraje es el que quedó tras la revisión. Si falta uno de los dos, la fila queda fuera de esta comparación. Para Pearson también hacemos un contraste bilateral. H₀ plantea una correlación lineal poblacional igual a cero. El valor p clásico requiere pares independientes y un modelo de normalidad conjunta bajo H₀. Usaremos el 5 % como umbral didáctico. Spearman se interpreta aquí como un coeficiente descriptivo.

In [ ]:
con_coche = base.loc[base['tiene_coche'].eq(True).fillna(False)]
pares = con_coche[['salario_eur', 'km_limpios']].dropna().astype(float)

print(f'Personas que declaran tener coche: {len(con_coche)}')
print(f'Pares completos: {len(pares)} · Sin algún dato: {len(con_coche) - len(pares)}')

if len(pares) < 2 or (pares.nunique() < 2).any():
    print('Para calcular la correlación hacen falta pares completos y variación en ambas medidas.')
else:
    salario = pares['salario_eur']
    km = pares['km_limpios']
    pearson = stats.pearsonr(salario, km)
    spearman = salario.rank().corr(km.rank())
    display(pd.Series({'Pearson': pearson.statistic, 'Spearman': spearman},
                      name='coeficiente').round(3))
    if len(pares) >= 3:
        print(f'Valor p bilateral de Pearson: {pearson.pvalue:.4g}')
        print('Decisión al 5 %:', 'rechazamos H₀' if pearson.pvalue < alpha else 'no rechazamos H₀')

**Cómo leerlo.** De 323 personas que declaran tener coche, tenemos 298 pares completos. Pearson da aproximadamente **−0,096** y Spearman **−0,070**. En esas filas, la relación lineal y la tendencia por orden son débiles. El valor p bilateral de Pearson ronda **0,099**. Con el umbral del 5 %, no rechazamos la hipótesis de correlación lineal cero bajo el modelo del contraste.

Un coeficiente cercano a cero puede ocultar una relación que cambia de dirección o diferencias entre grupos. También podrían influir la ciudad, el tipo de trabajo o quién dejó datos sin responder. Esta comparación es exploratoria. Por sí sola no permite concluir que el salario cambie los kilómetros recorridos.

[Pearson](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.pearsonr.html) · [Spearman](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.spearmanr.html)

## 9 · Del portátil a Big Data

En este cuaderno la encuesta cabe en memoria. En el [08 en PySpark](08_data_preparation_and_analysis.ipynb) puedes comparar cada resultado sobre los mismos archivos. Si tuviéramos muchos archivos o un histórico grande, podríamos guardar el Parquet en HDFS o S3 y usar Spark para contar nulos, detectar repeticiones, validar uniones y calcular resúmenes por grupo. Para Welch bastan el tamaño, la media y la desviación típica de cada grupo. Para chi-cuadrado basta la tabla de frecuencias. Así podemos trasladar al portátil solo esos resúmenes para el contraste.

**Fuentes.** Adaptación del nivel 1 y de la unión por ciudad del nivel 2 de `sprint_8.ipynb`. [NIST, análisis exploratorio](https://itl.nist.gov/div898/handbook/eda/section1/eda11.htm). [SciPy, Welch](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html). [SciPy, chi-cuadrado](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.contingency.chi2_contingency.html).

[Volver al README](../README.md) · [Siguiente: Visualización de datos](10_data_visualization.ipynb)